# Content-based Filtering

### NYC Merged Dataset

In [6]:
# Import libraries
import json
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Load final merged dataset
df_final = pd.read_csv("../data/processed/restaurants_data_merged_final.csv")

df_final.head()

,name,price,rating,review_count,cuisine,address,display_phone,comments
0,123 Burger Shot Beer,1,3.0,1000.0,"['american', 'sportsbars', 'tradamerican', 'ch...","738 10th Ave, Hells Kitchen, NY 10019",(212) 315-0123,NaN
1,One Stop Patty Shop,1,4.0,40.0,"['bakery', 'caribbean', 'breakfast_brunch']","1708 Amsterdam Ave, Harlem, NY 10031",(212) 491-7466,NaN
2,108 Food Dried Hot Pot,2,3.5,139.0,"['chinese', 'hotpot']","2794 Broadway, East Harlem, NY 10025",(917) 675-6878,NaN
3,Cookshop,2,4.0,1000.0,"['american', 'newamerican', 'breakfast_brunch'...","156 10th Ave, Midtown West, NY 10011",(212) 924-4440,NaN
4,11 Hanover Greek,3,4.0,122.0,"['greek', 'seafood', 'wine_bars']","11 Hanover Sq, Tribeca, NY 10005",(212) 785-4000,NaN


In [4]:
# Extract neighborhood from address
def extract_neighborhood(address):
    try:
        parts = address.split(",")
        if len(parts) >= 2:
            return parts[1].strip()
        return ""
    except:
        return ""

df_final["neighborhood"] = df_final["address"].apply(extract_neighborhood)

In [5]:
# Convert price and rating to text
def price_to_text(p):
    if p == 1:
        return "cheap"
    elif p == 2:
        return "moderate"
    elif p >= 3:
        return "expensive"
    return ""

def rating_to_text(r):
    if r >= 4:
        return "high_rating"
    elif r >= 3:
        return "medium_rating"
    else:
        return "low_rating"

df_final["price_text"] = df_final["price"].apply(price_to_text)
df_final["rating_text"] = df_final["rating"].apply(rating_to_text)

In [6]:
# Fill missing comments with empty string
df_final["comments"] = df_final["comments"].fillna("")

In [7]:
# Clean cuisine column 
def clean_cuisine(c):
    try:
        c_list = ast.literal_eval(c)
        return " ".join(c_list)
    except:
        return ""

df_final["cuisine_clean"] = df_final["cuisine"].apply(clean_cuisine)

# optional improvement
df_final["cuisine_clean"] = df_final["cuisine_clean"].str.replace("_", " ")

In [8]:
# Create combined features for content-based filtering
df_final["features"] = (
    df_final["cuisine_clean"] + " " +
    df_final["neighborhood"] + " " +
    df_final["price_text"] + " " +
    df_final["rating_text"] + " " +
    df_final["comments"]
)

In [9]:
# Vectorize features using TF-IDF
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(df_final["features"])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [10]:
# Create a mapping of restaurant names to indices
indices = pd.Series(df_final.index, index=df_final["name"]).drop_duplicates()

# Recommendation function
def recommend_restaurants(name, top_n=5):
    
    if name not in indices:
        return "Restaurant not found"
    
    idx = indices[name]
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+20]
    
    results = []
    
    for i, sim_score in sim_scores:
        row = df_final.iloc[i]
        
        final_score = (
            sim_score * 0.6 +
            (row["rating"] / 5) * 0.3 +
            (row["review_count"] / df_final["review_count"].max()) * 0.1
        )
        
        results.append((i, final_score))
    
    results = sorted(results, key=lambda x: x[1], reverse=True)[:top_n]
    
    return df_final.iloc[[i[0] for i in results]][["name","cuisine_clean","price","rating"]]

In [11]:
df_final["features"] = df_final["features"].str.lower()
df_final["features"] = df_final["features"].str.replace("_", " ")

In [12]:
# Test the recommendation function with top 3 recommendations
recommend_restaurants("Ming's Caffe", top_n=3)

,name,cuisine_clean,price,rating
2748,S Wan Cafe 洋紫荆,chinese hkcafe,1,4.5
1584,Joe’s Steam Rice Roll,chinese hkcafe hotdogs cantonese,1,4.0
1681,King's Kitchen,chinese,1,4.0


In [13]:
# Test the recommendation function with top 5 recommendations
recommend_restaurants("The Thirsty Koala", top_n=5)

,name,cuisine_clean,price,rating
3918,Trattoria L'incontro,italian vegetarian friendly vegan options,2,4.5
4361,Burke & Wills,australian,2,5.0
2734,Ruby's Cafe,australian bars breakfast brunch,2,5.0
4013,Martha's Country Bakery,bakeries cafe,2,4.5
3912,Osteria 166,italian vegetarian friendly vegan options,2,4.5


In [14]:
# Test the recommendation function with top 10 recommendations
recommend_restaurants("212 Steakhouse", top_n=10)

,name,cuisine_clean,price,rating
1044,Empire Steak House,american steak wine bars seafood,3,4.0
2522,Pietro's,italian steak wine bars,3,4.0
336,Benjamin Steakhouse,steak seafood,4,4.0
3191,Tempura Matsui,japanese seafood,4,4.5
2711,Rocco Steakhouse,steak tradamerican seafood,3,4.5
3644,Wokuni,japanese seafood,3,4.0
2999,Steak 'N Lobster,steak bars seafood,2,4.0
3505,Tuttles,american bars seafood,2,3.5
458,Bowery Meat Company,american steak wine bars,3,4.0
1913,Luke's Lobster Midtown East,seafood,2,4.0


# Evaluation

In [15]:
# Function to calculate cuisine overlap score for a given restaurant
def cuisine_overlap_score(base_name, top_n=5):
    
    recs = recommend_restaurants(base_name, top_n)
    
    base_cuisine = df_final[df_final["name"] == base_name]["cuisine_clean"].values[0]
    base_set = set(base_cuisine.split())
    
    scores = []
    
    for _, row in recs.iterrows():
        rec_set = set(row["cuisine_clean"].split())
        
        if len(base_set) == 0:
            scores.append(0)
        else:
            overlap = len(base_set & rec_set) / len(base_set)
            scores.append(overlap)
    
    return np.mean(scores)

In [16]:
# Test the cusine overlap score function
print("Ming's Caffe cuisine overlap score:", cuisine_overlap_score("Ming's Caffe", top_n=3))
print("The Thirsty Koala cuisine overlap score:", cuisine_overlap_score("The Thirsty Koala", top_n=5))
print("212 Steakhouse cuisine overlap score:", cuisine_overlap_score("212 Steakhouse", top_n=10))

Ming's Caffe cuisine overlap score: 0.8333333333333334
The Thirsty Koala cuisine overlap score: 0.4
212 Steakhouse cuisine overlap score: 0.55


In [17]:
# Function to calculate average rating score for recommended restaurants
def avg_rating_score(base_name, top_n=5):
    recs = recommend_restaurants(base_name, top_n)
    return recs["rating"].mean()

print("Ming's Caffe average rating score:", avg_rating_score("Ming's Caffe", top_n=3))
print("The Thirsty Koala average rating score:", avg_rating_score("The Thirsty Koala", top_n=5))
print("212 Steakhouse average rating score:", avg_rating_score("212 Steakhouse", top_n=10))

Ming's Caffe average rating score: 4.166666666666667
The Thirsty Koala average rating score: 4.7
212 Steakhouse average rating score: 4.05


In [18]:
# Function to calculate diversity score for recommended restaurants
def diversity_score(base_name, top_n=5):
    
    recs = recommend_restaurants(base_name, top_n)
    
    cuisines = recs["cuisine_clean"].tolist()
    unique = len(set(cuisines))
    
    return unique / len(cuisines)

print("Ming's Caffe diversity score:", diversity_score("Ming's Caffe", top_n=3))
print("The Thirsty Koala diversity score:", diversity_score("The Thirsty Koala", top_n=5))
print("212 Steakhouse diversity score:", diversity_score("212 Steakhouse", top_n=10))

Ming's Caffe diversity score: 1.0
The Thirsty Koala diversity score: 0.8
212 Steakhouse diversity score: 0.9


| Metric             | Good Value |
| ------------------ | ---------- |
| Cuisine similarity | 0.4 – 0.8  |
| Avg rating         | > 3.8      |
| Diversity          | 0.4 – 0.7  |


In [19]:
# Function to calculate precision@k for recommended restaurants
def precision_at_k(base_name, k=5):
    
    recs = recommend_restaurants(base_name, k)
    
    base_cuisine = df_final[df_final["name"] == base_name]["cuisine_clean"].values[0]
    
    relevant = 0
    
    for _, row in recs.iterrows():
        if any(c in row["cuisine_clean"] for c in base_cuisine.split()):
            relevant += 1
    
    return relevant / k

| Precision@K | Meaning                                    |
| ----------- | ------------------------------------------ |
| 1.0         | Perfect (all recommendations are relevant) |
| 0.8         | Very strong                                |
| 0.6         | Good                                       |
| 0.4         | Weak                                       |
| < 0.3       | Poor                                       |


In [20]:
# Precision@k for top 5 recommendations
print("Ming's Caffe precision@5:", precision_at_k("Ming's Caffe", k=5))
print("The Thirsty Koala precision@5:", precision_at_k("The Thirsty Koala", k=5))
print("212 Steakhouse precision@5:", precision_at_k("212 Steakhouse", k=5))

Ming's Caffe precision@5: 1.0
The Thirsty Koala precision@5: 0.8
212 Steakhouse precision@5: 1.0


### Yelp California Dataset

In [37]:
MAX_BUSINESSES = 5000
MAX_REVIEWS = 50000
MAX_USERS = 10000

business_file = "../data/external/yelp_academic_dataset_business.json"
review_file = "../data/external/yelp_academic_dataset_review.json"
user_file = "../data/external/yelp_academic_dataset_user.json"

In [38]:
# Load business data for CA restaurants
ca_businesses = []

with open(business_file, "r") as f:
    for line in f:
        row = json.loads(line)

        if row["state"] == "CA" and row["categories"] and "Restaurant" in row["categories"]:
            ca_businesses.append(row)

        if len(ca_businesses) >= MAX_BUSINESSES:
            break

business_df = pd.DataFrame(ca_businesses)

print("CA restaurants:", len(business_df))

CA restaurants: 1161


In [39]:
# Create a set of business IDs for CA restaurants
ca_ids = set(business_df["business_id"])

In [40]:
# Load reviews for the selected businesses
reviews = []

with open(review_file, "r") as f:
    for line in f:
        row = json.loads(line)

        if row["business_id"] in ca_ids:
            reviews.append(row)

        if len(reviews) >= MAX_REVIEWS:
            break

review_df = pd.DataFrame(reviews)

print("Filtered reviews:", len(review_df))

Filtered reviews: 50000


In [41]:
# Load user data for users who have reviewed the selected businesses
user_ids = set(review_df["user_id"])

users = []

with open(user_file, "r") as f:
    for line in f:
        row = json.loads(line)

        if row["user_id"] in user_ids:
            users.append(row)

        if len(users) >= MAX_USERS:
            break

user_df = pd.DataFrame(users)

print("Filtered users:", len(user_df))

Filtered users: 10000


In [42]:
# Merge business and review data
df = pd.merge(business_df, review_df, on="business_id")

df = df.rename(columns={"stars_x": "stars"})

In [43]:
# Export merged data for content-based filtering to processed folder
df.to_csv("../data/processed/content_based_data.csv", index=False)


In [ ]:
# Create combined features for content-based filtering
df["cuisine"] = df["categories"].fillna("").str.replace(",", " ")
df["location"] = df["city"]
df["review_text"] = df["text"].fillna("")


In [65]:
import ast

# Group by business_id and aggregate features
df_grouped = df.groupby("business_id").agg({
    "name": "first",
    "cuisine": "first",
    "stars": "first",
    "text": " ".join,
    "attributes": "first",
    "address": "first",
    "city": "first"
}).reset_index()


# Extract price from attributes
def extract_price(attr):
    try:
        if isinstance(attr, str):
            attr_dict = ast.literal_eval(attr)
            return int(attr_dict.get("RestaurantsPriceRange2", 0))
    except:
        return None
    return None

df_grouped["price"] = df_grouped["attributes"].apply(extract_price)


# Convert price to text
def price_to_text(p):
    if p == 1:
        return "cheap"
    elif p == 2:
        return "moderate"
    elif p in [3, 4]:
        return "expensive"
    return ""

df_grouped["price_text"] = df_grouped["price"].apply(price_to_text)

# Extract neighborhood from address
def extract_neighborhood(address):
    try:
        return address.split(",")[1].strip()
    except:
        return ""

df_grouped["neighborhood"] = df_grouped["address"].apply(extract_neighborhood)


df_grouped["location"] = df_grouped["city"]

# Create combined features for content-based filtering
df_grouped["features"] = (
    df_grouped["cuisine"].fillna("") + " " +
    df_grouped["price_text"].fillna("") + " " +
    df_grouped["neighborhood"].fillna("") + " " +
    df_grouped["location"].fillna("") + " " +
    df_grouped["text"].fillna("")
)

In [66]:
# Vectorize features using TF-IDF and compute cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2)
)

tfidf_matrix = tfidf.fit_transform(df_grouped["features"])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [67]:
# Create a mapping of restaurant names to indices
indices = pd.Series(df_grouped.index, index=df_grouped["name"]).groupby(level=0).first()

# Recommendation function
def recommend_restaurants(name, top_n=5):
    
    if name not in indices:
        return "Restaurant not found"
    
    idx = indices[name]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    
    restaurant_indices = [i[0] for i in sim_scores]
    
    return df_grouped[["name","cuisine","stars"]].iloc[restaurant_indices]

In [68]:
# Test the recommendation function with a non-existent restaurant
recommend_restaurants("Fake Restaurant")

'Restaurant not found'

In [69]:
# Test the recommendation function with a real restaurant
recommend_restaurants("Helena Avenue Bakery", top_n=5)

,name,cuisine,stars
94,Summerland Beach Cafe,Restaurants American (Traditional) Breakfast...,4.0
213,Our Daily Bread,Restaurants Cafes Breakfast & Brunch Bakeri...,3.5
304,Cajun Kitchen Cafe,Restaurants Breakfast & Brunch Cafes Cajun/...,4.0
140,Breakfast Culture Club,Nightlife Coffee & Tea Art Galleries Bagels...,4.0
47,The Worker Bee Cafe,Breakfast & Brunch Diners Restaurants Ameri...,4.0


In [71]:
# Test the recommendation function with a real restaurant
recommend_restaurants("Choppa Poke", top_n=3)

,name,cuisine,stars
319,PokiRito,Hawaiian Food Sushi Bars Poke Seafood Res...,4.0
153,Poke Theory SB,Coffee & Tea Noodles Restaurants Seafood J...,4.5
37,PokeCeviche,Asian Fusion Poke Seafood Food Restaurants,4.0


### Precision@K

In [72]:
def precision_at_k(name, k=5):
    
    if name not in indices:
        return 0
    
    recs = recommend_restaurants(name, top_n=k)
    
    if isinstance(recs, str):
        return 0
    
    target_cuisine = df_grouped[df_grouped["name"] == name]["cuisine"].values[0]
    
    relevant = 0
    
    for c in recs["cuisine"]:
        # check overlap in cuisine words
        if any(word in c for word in target_cuisine.split()):
            relevant += 1
    
    return relevant / k

In [73]:
# Test the recommendation function
sample_names = df_grouped["name"].sample(10)

scores = [precision_at_k(name, 5) for name in sample_names]

print("Average Precision@5:", sum(scores)/len(scores))

Average Precision@5: 1.0


In [75]:
# Recall@k 
def recall_at_k(name, k=5):
    if name not in indices:
        return 0
    
    recs = recommend_restaurants(name, k)
    
    target_cuisine = df_grouped[df_grouped["name"] == name]["cuisine"].values[0]
    
    relevant_total = sum(
        df_grouped["cuisine"].str.contains(target_cuisine.split()[0])
    )
    
    relevant_found = sum(
        any(word in c for word in target_cuisine.split()) for c in recs["cuisine"]
    )
    
    if relevant_total == 0:
        return 0
    
    return relevant_found / relevant_total

In [76]:
# Test recall@k
scores = [recall_at_k(name, 5) for name in sample_names]
print("Average Recall@5:", sum(scores)/len(scores))

Average Recall@5: 0.6480912452676106


In [ ]:
#F1-Score@K  
def f1_score_at_k(name, k=5):
    p = precision_at_k(name, k)
    r = recall_at_k(name, k)
    
    if p + r == 0:
        return 0
    
    return 2 * (p * r) / (p + r)

In [ ]:
# Test F1-Score@K
scores = [f1_score_at_k(name, 5) for name in sample_names]
print("Average F1-Score@5:", sum(scores)/len(scores))

Average F1-Score@5: 0.3826339649069039


In [ ]:
# Diversity@K - percentage of unique cuisines in the recommended list
def diversity_at_k(name, k=5):
    recs = recommend_restaurants(name, k)
    
    cuisines = recs["cuisine"]
    
    return len(set(cuisines)) / len(cuisines)

In [80]:
# Test Diversity@K
scores = [diversity_at_k(name, 5) for name in sample_names]
print("Average Diversity@5:", sum(scores)/len(scores))

Average Diversity@5: 0.9800000000000001


In [82]:
# Average rating of recommended restaurants
def avg_rating_at_k(name, k=5):
    recs = recommend_restaurants(name, k)
    return recs["stars"].mean()

In [83]:
# Test avg rating of recommended restaurants
scores = [avg_rating_at_k(name, 5) for name in sample_names]
print("Average rating of recommended restaurants @5:", sum(scores)/len(scores))

Average rating of recommended restaurants @5: 3.6700000000000004


In [84]:
# Coverage@K - percentage of unique restaurants recommended across all users
def coverage(sample_size=50):
    recommended = set()
    
    sample = df_grouped["name"].sample(sample_size)
    
    for name in sample:
        recs = recommend_restaurants(name, 5)
        recommended.update(recs["name"])
    
    return len(recommended) / len(df_grouped)

In [92]:
# Run all metrics together for a sample of restaurants
names = df_grouped["name"].sample(10)

print("Precision:", np.mean([precision_at_k(n) for n in names]))
print("Recall:", np.mean([recall_at_k(n) for n in names]))
print("F1:", np.mean([f1_score_at_k(n) for n in names]))
print("Diversity:", np.mean([diversity_at_k(n) for n in names]))
print("Avg Rating:", np.mean([avg_rating_at_k(n) for n in names]))
print("Coverage:", coverage(sample_size=10))

Precision: 1.0
Recall: 1.0519041939528155
F1: 0.6809290492365714
Diversity: 1.0
Avg Rating: 3.7600000000000002
Coverage: 0.10778443113772455
